# HF LLM: Natural Language → SQL (InferenceClient, schema as paragraph)

This notebook:
1. Reads your SQLite database `meta_ads.db`
2. Displays the schema in a readable, multi-line format
3. Uses Hugging Face `InferenceClient` to convert natural-language questions into SQLite SQL

Note: it generates SQL only (no SQL execution).

## 1) Setup (HF token + dependency)

You must set `HF_TOKEN` in your environment before running this notebook.

In [ ]:
%pip install -q -U huggingface_hub

In [1]:
import os

HF_TOKEN = os.environ.get("HF_TOKEN")

print("HF_TOKEN present:", bool(HF_TOKEN))
if not HF_TOKEN:
    raise RuntimeError("Missing HF_TOKEN. Set it in your environment, then restart the kernel.")
print("HF_TOKEN prefix:", HF_TOKEN[:8])

HF_TOKEN present: True
HF_TOKEN prefix: hf_Fgwth


## 2) Connect to SQLite + display schema (multi-line)

We convert the schema into a prompt-ready string called `SCHEMA`.

In [2]:
import sqlite3
from pathlib import Path

DB_PATH = Path("meta_ads.db")
assert DB_PATH.exists(), f"DB not found at {DB_PATH.resolve()}"

conn = sqlite3.connect(DB_PATH)

def get_tables(conn: sqlite3.Connection):
    rows = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"
    ).fetchall()
    return [r[0] for r in rows]

def get_table_info(conn: sqlite3.Connection, table: str):
    # PRAGMA table_info returns: cid, name, type, notnull, dflt_value, pk
    return conn.execute(f"PRAGMA table_info({table})").fetchall()

def build_schema_paragraph(conn: sqlite3.Connection) -> str:
    parts = []
    for t in get_tables(conn):
        cols = get_table_info(conn, t)
        parts.append(f"TABLE {t}:")
        for c in cols:
            col_name, col_type = c[1], c[2]
            parts.append(f"  - {col_name} {col_type}")
        parts.append("")
    return "\n".join(parts).strip()

SCHEMA = build_schema_paragraph(conn)
print(SCHEMA)

TABLE ad_products:
  - product_id INTEGER
  - product_name TEXT
  - placement TEXT
  - objective TEXT
  - pricing_model TEXT
  - active_from DATE

TABLE ad_transactions:
  - transaction_id INTEGER
  - event_ts TEXT
  - advertiser_id INTEGER
  - product_id INTEGER
  - campaign_id TEXT
  - currency TEXT
  - impressions INTEGER
  - clicks INTEGER
  - conversions INTEGER
  - spend_usd REAL
  - revenue_usd REAL

TABLE advertisers:
  - advertiser_id INTEGER
  - advertiser_name TEXT
  - industry TEXT
  - hq_country TEXT
  - account_tier TEXT
  - created_at DATE


## 3) HF InferenceClient call (chat.completions)

This uses the exact call style you tested in `test_llm.ipynb`.

In [4]:
from huggingface_hub import InferenceClient
import tqdm as notebook_tqdm
# Keep the same model as your working test_llm.ipynb
HF_MODEL_ID = os.environ.get("HF_MODEL_ID") or "google/gemma-3-27b-it"
print("Using model:", HF_MODEL_ID)

client = InferenceClient(api_key=HF_TOKEN)

def hf_generate(prompt: str, max_new_tokens: int = 256, temperature: float = 0.0) -> str:
    messages = [{"role": "user", "content": prompt}]
    response = client.chat.completions.create(
        model=HF_MODEL_ID,
        messages=messages,
        max_tokens=max_new_tokens,
        temperature=temperature,
    )
    return response.choices[0].message.content

Using model: google/gemma-3-27b-it


## 4) Prompt template (schema + few-shot)

We ask the model to output ONLY a single SQLite `SELECT` statement.

In [5]:
import re
from textwrap import indent

FEW_SHOT = """
Example 1
Question: What is total revenue?
SQL: SELECT SUM(t.revenue_usd) AS total_revenue_usd FROM ad_transactions t;

Example 2
Question: Revenue by placement
SQL: SELECT p.placement, SUM(t.revenue_usd) AS revenue_usd
     FROM ad_transactions t
     JOIN ad_products p ON p.product_id = t.product_id
     GROUP BY p.placement
     ORDER BY revenue_usd DESC;
""".strip()

PROMPT_TEMPLATE = """
You are an expert data analyst.
Given a SQLite database schema, write ONE SQLite SQL query that answers the question.

Rules:
- Output ONLY the SQL query.
- Must be a single SELECT statement.
- Do NOT use ``` fences.
- Use only table/column names that exist in the schema.

Schema:
{schema}

{few_shot}

Question: {question}
SQL:
""".strip()

def build_prompt(question: str) -> str:
    return PROMPT_TEMPLATE.format(schema=SCHEMA, few_shot=FEW_SHOT, question=question)

def extract_first_select(text: str) -> str:
    m = re.search(r"\bselect\b[\s\S]*", text, flags=re.IGNORECASE)
    if not m:
        return text.strip()
    sql = m.group(0).strip()
    sql = re.sub(r"```[\s\S]*?```", "", sql).strip()
    return sql


## 5) NL → SQL examples (easy → hard)

This prints the generated SQL for each natural-language query.

In [6]:
questions = [
    "What is total revenue?",
    "What is overall ROAS?",
    "Top 10 advertisers by revenue",
    "Revenue by placement",
    "Daily revenue last 30 days",
    "Top 5 products by ROAS last 60 days",
    "Revenue by industry for Enterprise advertisers last 90 days",
    "Top 10 advertisers by revenue for placement = Reels last 45 days",
]

for q in questions:
    prompt = build_prompt(q)
    raw = hf_generate(prompt, max_new_tokens=256, temperature=0.0)
    sql = extract_first_select(raw)
    print("NL:", q)
    print("SQL:\n" + indent(sql, "  "))
    print("-" * 80)


NL: What is total revenue?
SQL:
  SELECT SUM(revenue_usd) AS total_revenue FROM ad_transactions;
--------------------------------------------------------------------------------
NL: What is overall ROAS?
SQL:
  SELECT SUM(revenue_usd) / SUM(spend_usd) AS overall_roas FROM ad_transactions;
--------------------------------------------------------------------------------
NL: Top 10 advertisers by revenue
SQL:
  SELECT a.advertiser_name, SUM(t.revenue_usd) AS total_revenue_usd
  FROM ad_transactions t
  JOIN advertisers a ON t.advertiser_id = a.advertiser_id
  GROUP BY a.advertiser_name
  ORDER BY total_revenue_usd DESC
  LIMIT 10;
--------------------------------------------------------------------------------
NL: Revenue by placement
SQL:
  SELECT p.placement, SUM(t.revenue_usd) AS revenue_usd
       FROM ad_transactions t
       JOIN ad_products p ON p.product_id = t.product_id
       GROUP BY p.placement
       ORDER BY revenue_usd DESC;
------------------------------------------------

## 6) Cleanup

In [ ]:
import pandas as pd

query = """
SELECT
  a.advertiser_name,
  SUM(t.revenue_usd) AS total_revenue
FROM ad_transactions AS t
JOIN ad_products AS p
  ON t.product_id = p.product_id
JOIN advertisers AS a
  ON t.advertiser_id = a.advertiser_id
WHERE
  p.placement = 'Reels' AND t.event_ts >= DATE('now', '-45 days')
GROUP BY
  a.advertiser_name
ORDER BY
  total_revenue DESC
LIMIT 10;
""".strip()

df = pd.read_sql_query(query, conn)
df

conn.close()